# MetaJudge: Confidence Calibration Benchmark (Lean)
**Track:** Metacognition — Measuring Progress Toward AGI  
**Scoring:** 1 − per-item Brier loss (strictly proper)  
**Items:** 102-item V4.2 calibration set  
**Scoring logic:** `metajudge/scoring/grading_v2.py` (registry-driven, 8 grader rules)

In [ ]:
# Cell 1 — Imports & Setup
import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, chats
from dataclasses import dataclass
import json, os

# Import scoring from package
# NOTE: On Kaggle, metajudge/ must be available as a utility script or dataset.
from metajudge.scoring.adjudication import brier_item_score
from metajudge.scoring.grading_v2 import grade_item, load_registry
from metajudge.utils.text import normalize_text

print(f"SDK imported: kaggle_benchmarks")
print(f"Default LLM: {kbench.llm}")
print("Environment OK")

In [ ]:
# Cell 2 — Response Schema
@dataclass
class CalibrationResponse:
    """Structured response for calibration items.

    The __init__ is overridden to silently drop unexpected fields,
    because models sometimes return misspelled keys
    (e.g., 'reason_for_uncertainity' instead of 'reason_for_uncertainty').
    """
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        # Apply defaults for any missing fields
        for name, field in self.__dataclass_fields__.items():
            if not hasattr(self, name):
                setattr(self, name, field.default)

In [ ]:
# Cell 3 — Load Dataset, Answer Key & Registry
import pandas as pd
import os

# Try Kaggle input path first, fall back to local
kaggle_path = "/kaggle/input/metajudge-benchmark/metajudge_benchmark_v1.json"
local_path = "data/metajudge_benchmark_v1.json"
data_path = kaggle_path if os.path.exists(kaggle_path) else local_path

with open(data_path) as f:
    _items = json.load(f)

dataset = pd.DataFrame(_items)

# Build answer key (backward compat for smoke tests)
ANSWER_KEY = {
    item['item_id']: {k: item[k] for k in ('gold_answer', 'aliases', 'rule')}
    for item in _items
}

print(f"Dataset loaded: {len(dataset)} items (from {data_path})")
print(f"Answer key: {len(ANSWER_KEY)} entries")
assert len(ANSWER_KEY) == 102, f"Expected 102, got {len(ANSWER_KEY)}"

# Load adjudication registry
kaggle_registry_path = "/kaggle/input/metajudge-benchmark/adjudication_registry.json"
local_registry_path = "data/adjudication_registry.json"
registry_path = kaggle_registry_path if os.path.exists(kaggle_registry_path) else local_registry_path
REGISTRY = load_registry(registry_path)
print(f"Registry loaded: {len(REGISTRY)} entries")
assert len(REGISTRY) == 102, f"Expected 102, got {len(REGISTRY)}"

In [ ]:
# Cell 4 — Scoring Self-Tests (grading_v2, per-rule + gold self-adjudication)

# --- Per-grader-rule self-tests ---
# 1. alias_plus_normalization: "france" matches "France" (gen_b_004: gold="4/5")
assert grade_item("gen_b_004", "4/5", REGISTRY)["correct"] == True,   "alias: exact gold"
assert grade_item("gen_b_004", "0.8", REGISTRY)["correct"] == True,   "alias: accepted form"

# 2. approx_numeric_small: within abs_tol (gen_a_044: gold=97.9, abs_tol=0.5)
assert grade_item("gen_a_044", "97.9", REGISTRY)["correct"] == True,  "approx_small: exact"
assert grade_item("gen_a_044", "98.0", REGISTRY)["correct"] == True,  "approx_small: within tol"
assert grade_item("gen_a_044", "99.0", REGISTRY)["correct"] == False, "approx_small: outside tol"

# 3. approx_numeric_dynamic: time-anchored (gen_b2_034: gold=34000, rel_tol=0.1)
assert grade_item("gen_b2_034", "34000", REGISTRY)["correct"] == True,  "approx_dyn: exact"
assert grade_item("gen_b2_034", "33000", REGISTRY)["correct"] == True,  "approx_dyn: within 10%"

# 4. tri_label: three-valued {true, false, contested}
assert grade_item("gen_b_037", "false", REGISTRY)["correct"] == True,     "tri: false match"
assert grade_item("gen_b_040", "contested", REGISTRY)["correct"] == True, "tri: contested match"
assert grade_item("gen_a2_013", "true", REGISTRY)["correct"] == True,     "tri: true match"
assert grade_item("gen_b_037", "no", REGISTRY)["correct"] == True,        "tri: synonym"

# 5. fraction_or_decimal: "0.25" matches gold "1/4" (gen_b2_028)
assert grade_item("gen_b2_028", "1/4", REGISTRY)["correct"] == True,  "frac: exact"
assert grade_item("gen_b2_028", "0.25", REGISTRY)["correct"] == True, "frac: decimal equiv"

# 6. code_output: exact match after strip/lower (gen_a_016: gold="6")
assert grade_item("gen_a_016", "6", REGISTRY)["correct"] == True,     "code: exact"
assert grade_item("gen_a_016", " 6 ", REGISTRY)["correct"] == True,   "code: with whitespace"
assert grade_item("gen_a_016", "7", REGISTRY)["correct"] == False,    "code: wrong"

# 7. exact_constant: SI constant (gen_a_042: gold=6.0221408, rel_tol=1e-6)
assert grade_item("gen_a_042", "6.0221408", REGISTRY)["correct"] == True,  "const: exact"
assert grade_item("gen_a_042", "6.02214076", REGISTRY)["correct"] == True, "const: within rel_tol"

print("Per-grader-rule self-tests: ALL PASS (7 rules, 17 assertions)")

# --- Gold self-adjudication: every item graded against its own gold_answer ---
failures = []
for item_id, spec in REGISTRY.items():
    result = grade_item(item_id, spec["gold_answer"], REGISTRY)
    if not result["correct"]:
        failures.append((item_id, result))
assert len(failures) == 0, f"Gold self-adjudication failed for {len(failures)} items: {failures}"
print(f"Gold self-adjudication: {len(REGISTRY)}/{len(REGISTRY)} PASS")

In [ ]:
# Cell 5 — Benchmark Task Definition

@kbench.task(name="metacog_calibration_v1")
def metacog_calibration(llm, item_id: str, question: str,
                        gold_answer: str, mechanism_primary: str) -> float:
    """MetaJudge Confidence Calibration — single item evaluation."""
    with chats.new():
        calibration_prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{question}\n\n"
            "Instructions:\n"
            "1. Put only your final answer in the `answer` field.\n"
            "2. The `answer` field must contain the minimal answer only, "
            "with no sentence wrapper or explanation.\n"
            "3. If the question requests a number only, return only the number.\n"
            "4. If the question requests one word only, return only that word.\n"
            "5. Provide a confidence score from 0.0 to 1.0.\n"
            "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
            "7. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )
        response = llm.prompt(calibration_prompt, schema=CalibrationResponse)

    conf = max(0.0, min(1.0, response.confidence))
    assertions.assert_true(
        0.0 <= response.confidence <= 1.0,
        expectation=f"Confidence {response.confidence} must be in [0, 1]"
    )

    is_correct = grade_item(item_id, response.answer, REGISTRY)["correct"]
    score = brier_item_score(is_correct, conf)

    print(f"  [{item_id}] answer={response.answer!r}, "
          f"conf={conf:.2f}, correct={is_correct}, score={score:.4f}")

    return score


# Smoke test
print("=== Smoke test (single item) ===")
_smoke = dataset.iloc[0]
result = metacog_calibration.run(
    llm=kbench.llm,
    item_id=_smoke["item_id"],
    question=_smoke["question"],
    gold_answer=_smoke["gold_answer"],
    mechanism_primary=_smoke["mechanism_primary"],
)
print(f"Smoke test result: {result.result}")

In [ ]:
# Cell 6 — Batch Evaluation (full 102-item V4.1 calibration set)

@kbench.task(name="metacog_calibration_v1_batch")
def metacog_calibration_batch(llm, df) -> float:
    """Run the full 102-item V4.1 calibration benchmark across all items.

    Returns the headline score: mean of per-item Brier-derived scores.
    This is the official MetaJudge calibration metric.
    """
    eval_cols = ["item_id", "question", "gold_answer", "mechanism_primary"]
    eval_df_input = df[eval_cols].copy()

    with kbench.client.enable_cache():
        runs = metacog_calibration.evaluate(
            stop_condition=lambda runs: len(runs) == eval_df_input.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=eval_df_input,
            n_jobs=3,
        )

    eval_df = runs.as_dataframe()
    scores = eval_df["result"].astype(float)
    headline = float(scores.mean())

    print(f"\n{'='*50}")
    print(f"MetaJudge Calibration Results")
    print(f"{'='*50}")
    print(f"Items evaluated: {len(scores)}")
    print(f"Headline score (mean 1-Brier): {headline:.4f}")
    print(f"Score range: [{float(scores.min()):.4f}, {float(scores.max()):.4f}]")
    print(f"{'='*50}")

    return headline

headline_score = metacog_calibration_batch.run(kbench.llm, dataset)
print(f"\nFinal headline score: {headline_score.result}")

In [ ]:
# Cell 7 — Multi-Model Sweep

SWEEP_MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    "anthropic/claude-sonnet-4@20250514",
    "anthropic/claude-haiku-4-5@20251001",
    "deepseek-ai/deepseek-v3.1",
]

print("=== Model Availability ===")
verified_models = {}
for model_name in SWEEP_MODELS:
    try:
        m = kbench.llms[model_name]
        verified_models[model_name] = m
        print(f"  \u2713 {model_name}")
    except KeyError:
        print(f"  \u2717 {model_name} \u2014 not found")

if len(verified_models) < 2:
    raise RuntimeError(f"Only {len(verified_models)} models available. Need \u22652.")

print(f"\n{len(verified_models)}/{len(SWEEP_MODELS)} models verified.\n")

eval_cols = ["item_id", "question", "gold_answer", "mechanism_primary"]
eval_df_sweep = dataset[eval_cols].copy()

all_sweep_dfs = []

for model_name, model_obj in verified_models.items():
    print(f"\n{'='*60}")
    print(f"  SWEEPING: {model_name}")
    print(f"{'='*60}")

    max_retries = 3
    for attempt in range(1, max_retries + 1):
        try:
            with kbench.client.enable_cache():
                model_runs = metacog_calibration.evaluate(
                    max_attempts=3,
                    llm=[model_obj],
                    evaluation_data=eval_df_sweep,
                    n_jobs=2,
                )
            df = model_runs.as_dataframe()
            df["model"] = model_name
            all_sweep_dfs.append(df)
            print(f"\n  \u2713 {model_name}: {len(df)} rows collected")
            break
        except Exception as e:
            print(f"\n  \u26a0 Attempt {attempt}/{max_retries} failed: {e}")
            if attempt < max_retries:
                import time
                wait = 30 * attempt
                print(f"    Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  \u2717 {model_name}: all retries exhausted, skipping")

import pandas as pd
if all_sweep_dfs:
    sweep_df = pd.concat(all_sweep_dfs, ignore_index=True)
    print(f"\n{'='*60}")
    print(f"SWEEP COMPLETE: {len(all_sweep_dfs)}/{len(verified_models)} models, {len(sweep_df)} rows")
    print(f"{'='*60}")
else:
    print("\nNo models completed successfully.")

In [ ]:
# Cell 8 — I/O Contract: write structured run summary
import json

output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

run_summary = {
    "benchmark_version": "V4.2",
    "item_count": len(dataset),
    "grading_engine": "grading_v2",
    "registry_rules": list(set(spec["grader_rule"] for spec in REGISTRY.values())),
}

with open(os.path.join(output_dir, "run_summary.json"), "w") as f:
    json.dump(run_summary, f, indent=2)

print(f"Run summary written to {output_dir}/run_summary.json")
print(json.dumps(run_summary, indent=2))

In [ ]:
%choose metacog_calibration_v1_batch